# LightMamba-ASL — Kaggle ASL Signs (Parquet Landmarks)

**Dataset:** Kaggle `asl-signs` — 250 signs, ~94k samples, pre-extracted MediaPipe parquet landmarks  
**No raw video needed** — parquet files already contain MediaPipe landmark coordinates  
**Model:** Same LightMamba-ASL architecture, landmark-only mode (RGB branch receives zeros; reliability fusion auto-prioritizes landmarks)

---
### Before running:
1. Runtime → Change runtime type → **T4 GPU**
2. Accept competition rules at https://www.kaggle.com/competitions/asl-signs
3. Kaggle → Profile → Settings → API → **Create New Token** → download `kaggle.json`
4. Run cells top to bottom

## Step 1 — Install Dependencies & Kaggle Auth

In [ ]:
!pip install -q kaggle pyarrow torch torchvision tqdm

from google.colab import files
print('Upload your kaggle.json file:')
files.upload()  # select kaggle.json

!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/kaggle.json
!chmod 600 ~/.kaggle/kaggle.json
print('Kaggle auth configured.')

## Step 2 — Download & Extract Kaggle Dataset

In [ ]:
import os

ROOT = '/content/asl_data'
os.makedirs(ROOT, exist_ok=True)

!kaggle competitions download -c asl-signs -p {ROOT}
!unzip -q {ROOT}/asl-signs.zip -d {ROOT}

print('Files:', os.listdir(ROOT))

## Step 3 — Mount Drive & Extract Project Code

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import zipfile, shutil, sys

PROJECT_ZIP = '/content/drive/MyDrive/second_review.zip'  # adjust path if needed
PROJECT_DIR = '/content/LightMamba'

if os.path.exists(PROJECT_DIR):
    shutil.rmtree(PROJECT_DIR)
os.makedirs(PROJECT_DIR, exist_ok=True)

with zipfile.ZipFile(PROJECT_ZIP, 'r') as z:
    z.extractall(PROJECT_DIR)

# Find actual project root (where backend/ lives)
project_root = PROJECT_DIR
for item in os.listdir(PROJECT_DIR):
    candidate = os.path.join(PROJECT_DIR, item)
    if os.path.isdir(candidate) and 'backend' in os.listdir(candidate):
        project_root = candidate
        break

os.chdir(project_root)
if project_root not in sys.path:
    sys.path.insert(0, project_root)

print('Project root:', project_root)
print('Contents:', os.listdir('.'))

## Step 4 — Verify GPU

In [ ]:
import torch
print('CUDA:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')

## Step 5 — Inspect Parquet Format
Understand the landmark columns before loading.

In [ ]:
import pandas as pd
import json

train_df = pd.read_csv(f'{ROOT}/train.csv')
print('Total samples:', len(train_df))
print('Total signs:', train_df['sign'].nunique())
print(train_df.head(3))

# Inspect one parquet file
sample = pd.read_parquet(f"{ROOT}/{train_df.iloc[0]['path']}")
print('\nParquet columns:', sample.columns.tolist())
print('Landmark types:', sample['type'].unique())
print('Frames:', sample['frame'].nunique())
print(sample.head(5))

## Step 6 — Landmark Loader

Maps Kaggle parquet → exact LightMamba-ASL input format:
- `left_hand` (21 pts) + `right_hand` (21 pts) + `pose` (33 pts) = **75 landmarks × 3 = 225 dims**
- Validity mask: `[T, 3]` — `[left_hand_present, right_hand_present, pose_present]`
- Normalization: same wrist-centered / shoulder-centered strategy as `landmark_normalizer.py`
- Motion features: same `compute_motion_features()` from `backend/features/motion_features.py`

In [ ]:
import numpy as np
from backend.features.motion_features import compute_motion_features
from backend.features.landmark_normalizer import normalize_landmarks
from backend.config import NUM_FRAMES

# Kaggle pose uses 543 landmarks total; we need indices 489-521 (33 pose landmarks)
# Reference: https://www.kaggle.com/competitions/asl-signs/data
POSE_LANDMARK_COUNT = 33
HAND_LANDMARK_COUNT = 21

def load_parquet_landmarks(row, max_frames=NUM_FRAMES):
    """
    Loads a Kaggle ASL Signs parquet file and returns:
      landmarks: [T, 225]  (75 landmarks × 3 coords, flattened)
      mask:      [T, 3]    (left_hand, right_hand, pose validity)
    Matches exactly the format expected by LandmarkBranch + MultimodalFusion.
    """
    df = pd.read_parquet(f"{ROOT}/{row['path']}")

    frames = sorted(df['frame'].unique())
    seq_landmarks = []
    seq_masks = []

    for frame_idx in frames:
        fdf = df[df['frame'] == frame_idx]

        lh_coords   = np.zeros((HAND_LANDMARK_COUNT, 3), dtype=np.float32)
        rh_coords   = np.zeros((HAND_LANDMARK_COUNT, 3), dtype=np.float32)
        pose_coords = np.zeros((POSE_LANDMARK_COUNT, 3), dtype=np.float32)
        mask_frame  = np.zeros(3, dtype=np.float32)

        lh = fdf[fdf['type'] == 'left_hand'].sort_values('landmark_index')
        if len(lh) == HAND_LANDMARK_COUNT:
            lh_coords = lh[['x', 'y', 'z']].values.astype(np.float32)
            mask_frame[0] = 1.0

        rh = fdf[fdf['type'] == 'right_hand'].sort_values('landmark_index')
        if len(rh) == HAND_LANDMARK_COUNT:
            rh_coords = rh[['x', 'y', 'z']].values.astype(np.float32)
            mask_frame[1] = 1.0

        pose = fdf[fdf['type'] == 'pose'].sort_values('landmark_index')
        if len(pose) == POSE_LANDMARK_COUNT:
            pose_coords = pose[['x', 'y', 'z']].values.astype(np.float32)
            mask_frame[2] = 1.0

        # Stack: [75, 3] — same order as LandmarkExtractor: lh, rh, pose
        combined = np.concatenate([lh_coords, rh_coords, pose_coords], axis=0)
        seq_landmarks.append(combined)
        seq_masks.append(mask_frame)

    # Temporal sampling: truncate or pad to NUM_FRAMES
    T = len(seq_landmarks)
    if T >= max_frames:
        # Uniform sampling
        indices = np.linspace(0, T - 1, max_frames, dtype=int)
        seq_landmarks = [seq_landmarks[i] for i in indices]
        seq_masks     = [seq_masks[i]     for i in indices]
    else:
        # Pad with zeros
        pad = max_frames - T
        seq_landmarks += [np.zeros((75, 3), dtype=np.float32)] * pad
        seq_masks     += [np.zeros(3, dtype=np.float32)]       * pad

    landmarks_3d = np.stack(seq_landmarks, axis=0)  # [T, 75, 3]
    mask_arr     = np.stack(seq_masks,     axis=0)  # [T, 3]

    # Apply same normalization as landmark_normalizer.py
    landmarks_3d = normalize_landmarks(landmarks_3d, mask_arr)

    # Flatten to [T, 225]
    landmarks_flat = landmarks_3d.reshape(max_frames, 225)

    # Compute motion features — same as dataset.py
    motion = compute_motion_features(landmarks_3d, mask_arr)  # [T, 225] or [T, 450]
    landmark_features = np.concatenate([landmarks_flat, motion], axis=-1)  # [T, 675] with 2nd order

    return landmark_features.astype(np.float32), mask_arr.astype(np.float32)


# Quick sanity check
lm, mk = load_parquet_landmarks(train_df.iloc[0])
print('Landmark features shape:', lm.shape)  # expected: (32, 675)
print('Mask shape:', mk.shape)               # expected: (32, 3)
print('Mask sample (frame 0):', mk[0])       # [left_hand, right_hand, pose]

## Step 7 — Build Label Map & Dataset

Kaggle has 250 signs. We train on all 250 (or a subset for quick testing).

In [ ]:
import json
from torch.utils.data import Dataset, DataLoader

with open(f'{ROOT}/sign_to_prediction_index_map.json') as f:
    sign_map = json.load(f)  # {sign_name: index}

index_to_sign = {v: k for k, v in sign_map.items()}
NUM_KAGGLE_CLASSES = len(sign_map)
print('Total Kaggle classes:', NUM_KAGGLE_CLASSES)

train_df['label'] = train_df['sign'].map(sign_map)

# --- Optional: subset for quick testing ---
# Uncomment next 2 lines to train on 20 signs only
# selected = train_df['sign'].unique()[:20]
# train_df = train_df[train_df['sign'].isin(selected)].reset_index(drop=True)

# Train/val split (80/20 stratified by sign)
from sklearn.model_selection import train_test_split
train_split, val_split = train_test_split(
    train_df, test_size=0.2, stratify=train_df['sign'], random_state=42
)
train_split = train_split.reset_index(drop=True)
val_split   = val_split.reset_index(drop=True)

print(f'Train: {len(train_split)} | Val: {len(val_split)}')


class KaggleASLDataset(Dataset):
    def __init__(self, df):
        self.df = df

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        lm_features, mask = load_parquet_landmarks(row)
        return {
            'landmarks': torch.tensor(lm_features, dtype=torch.float32),  # [T, 675]
            'mask':      torch.tensor(mask,        dtype=torch.float32),  # [T, 3]
            'label':     int(row['label']),
        }


train_dataset = KaggleASLDataset(train_split)
val_dataset   = KaggleASLDataset(val_split)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=32, shuffle=False, num_workers=2, pin_memory=True)

print('Loaders ready.')

## Step 8 — Build Landmark-Only Model

Kaggle dataset has no RGB video. We pass **zero RGB features** `[B, T, 960]` so the reliability fusion gate (alpha) learns to suppress RGB and rely entirely on landmarks (beta gate).

The classifier head is replaced to output 250 classes instead of 100.

In [ ]:
import torch
import torch.nn as nn
from backend.models.landmark_branch import LandmarkBranch
from backend.models.fusion import MultimodalFusion
from backend.models.hms_mamba import HMSMamba
from backend.config import MAMBA_HIDDEN_DIM, DROPOUT

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')


class LightMambaASL_LandmarkOnly(nn.Module):
    """
    LightMamba-ASL with landmark-only input.
    RGB branch is bypassed — zero tensor passed to fusion so reliability
    gate learns to weight landmarks exclusively.
    """
    def __init__(self, num_classes: int):
        super().__init__()
        self.landmark_branch = LandmarkBranch()   # [B,T,675] → [B,T,256]
        self.fusion          = MultimodalFusion()  # needs rgb_dim=960 zero input
        self.temporal_model  = HMSMamba()
        self.classifier = nn.Sequential(
            nn.LayerNorm(MAMBA_HIDDEN_DIM),
            nn.Dropout(DROPOUT),
            nn.Linear(MAMBA_HIDDEN_DIM, num_classes)
        )

    def forward(self, landmarks, mask):
        B, T, _ = landmarks.shape
        # Zero RGB features — fusion gate will suppress this branch
        rgb_zeros = torch.zeros(B, T, 960, device=landmarks.device)

        land_emb = self.landmark_branch(landmarks)          # [B, T, 256]
        fused    = self.fusion(rgb_zeros, land_emb, mask)   # [B, T, 256]
        temporal = self.temporal_model(fused)               # [B, 256]
        return self.classifier(temporal)                    # [B, num_classes]


model = LightMambaASL_LandmarkOnly(num_classes=NUM_KAGGLE_CLASSES).to(device)

# Dry run
with torch.no_grad():
    dummy_lm   = torch.randn(2, 32, 675).to(device)
    dummy_mask = torch.ones(2, 32, 3).to(device)
    out = model(dummy_lm, dummy_mask)
    print('Output shape:', out.shape)  # [2, 250]

total_params = sum(p.numel() for p in model.parameters())
print(f'Total parameters: {total_params:,}')

## Step 9 — Training Loop

In [ ]:
from tqdm import tqdm

EPOCHS    = 50
LR        = 1e-4
SAVE_PATH = '/content/kaggle_asl_best.pth'

optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-3)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
loss_fn   = nn.CrossEntropyLoss(label_smoothing=0.1)

best_val_acc = 0.0
patience, patience_counter = 10, 0

for epoch in range(1, EPOCHS + 1):
    # --- Train ---
    model.train()
    train_loss, correct, total = 0.0, 0, 0
    for batch in tqdm(train_loader, desc=f'Epoch {epoch}/{EPOCHS} [Train]', leave=False):
        lm     = batch['landmarks'].to(device)
        mask   = batch['mask'].to(device)
        labels = batch['label']
        labels = torch.tensor(labels) if not isinstance(labels, torch.Tensor) else labels
        labels = labels.to(device)

        optimizer.zero_grad()
        logits = model(lm, mask)
        loss   = loss_fn(logits, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        train_loss += loss.item() * labels.size(0)
        correct    += (logits.argmax(1) == labels).sum().item()
        total      += labels.size(0)

    train_acc = correct / total

    # --- Validate ---
    model.eval()
    val_loss, val_correct, val_total = 0.0, 0, 0
    with torch.no_grad():
        for batch in val_loader:
            lm     = batch['landmarks'].to(device)
            mask   = batch['mask'].to(device)
            labels = batch['label']
            labels = torch.tensor(labels) if not isinstance(labels, torch.Tensor) else labels
            labels = labels.to(device)

            logits    = model(lm, mask)
            val_loss += loss_fn(logits, labels).item() * labels.size(0)
            val_correct += (logits.argmax(1) == labels).sum().item()
            val_total   += labels.size(0)

    val_acc = val_correct / val_total
    scheduler.step()

    print(f'Epoch {epoch:02d}/{EPOCHS} | '
          f'Train Loss: {train_loss/total:.4f} | Train Acc: {train_acc*100:.2f}% | '
          f'Val Loss: {val_loss/val_total:.4f} | Val Acc: {val_acc*100:.2f}%')

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        patience_counter = 0
        torch.save({'epoch': epoch, 'model_state_dict': model.state_dict(),
                    'val_acc': val_acc, 'index_to_sign': index_to_sign}, SAVE_PATH)
        print(f'  ✓ Best model saved (val_acc={val_acc*100:.2f}%)')
    else:
        patience_counter += 1
        if patience_counter >= patience:
            print(f'Early stopping at epoch {epoch}.')
            break

print(f'\nTraining complete. Best val accuracy: {best_val_acc*100:.2f}%')

## Step 10 — Save to Google Drive

In [ ]:
import shutil

DRIVE_SAVE = '/content/drive/MyDrive/LightMamba_Kaggle_outputs'
os.makedirs(DRIVE_SAVE, exist_ok=True)

shutil.copy(SAVE_PATH, f'{DRIVE_SAVE}/kaggle_asl_best.pth')
print('Checkpoint saved to Drive:', DRIVE_SAVE)

## Step 11 — Quick Inference Test
Predict the sign for a single sample from the validation set.

In [ ]:
# Load best checkpoint
ckpt = torch.load(SAVE_PATH, map_location=device)
model.load_state_dict(ckpt['model_state_dict'])
model.eval()

sample_row = val_split.iloc[0]
lm, mask = load_parquet_landmarks(sample_row)

with torch.no_grad():
    logits = model(
        torch.tensor(lm).unsqueeze(0).to(device),
        torch.tensor(mask).unsqueeze(0).to(device)
    )
    probs = torch.softmax(logits, dim=-1)[0]
    top5  = probs.topk(5)

print('Ground truth:', sample_row['sign'])
print('Top-5 predictions:')
for score, idx in zip(top5.values.cpu(), top5.indices.cpu()):
    print(f'  {index_to_sign[idx.item()]:30s}  {score.item()*100:.1f}%')